# 08 - Persistencia en Firebase / Firestore

Este notebook documenta y prepara la carga de metadata del dataset, metricas de modelos, resultados comparativos y configuraciones experimentales en Firestore. No reentrena modelos y no modifica archivos en `data/raw/`.

## Por que Firestore

Firestore es una base NoSQL orientada a documentos. Es adecuada para esta etapa porque los resultados del proyecto tienen estructura semiestructurada: metadata del dataset, metricas por experimento, configuraciones de entrenamiento y resultados de validacion cruzada. Estos documentos pueden evolucionar sin exigir un esquema relacional rigido.

Persistir metricas y configuracion es clave para la trazabilidad del experimento. No alcanza con saber cual fue el mejor modelo: tambien se necesita guardar con que dataset, features, target, particion, pesos de clase y criterio de seleccion se obtuvo ese resultado.

## Colecciones creadas

- `datasets`: metadata del dataset procesado `siniestros_limpio_enriquecido`.
- `model_results`: un documento por experimento/modelo con metricas principales.
- `model_config`: un documento por configuracion experimental con features, preprocessing, split y notas metodologicas.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.upload_results_firebase import (
    DATA_FILE,
    LOG_FILE,
    build_firestore_payloads,
    upload_payloads,
)
from src.firebase_client import get_pipeline_logger, load_env_file

pd.set_option("display.max_columns", 100)

## Revision de metadata local

Antes de subir a Firestore se lee el CSV procesado para obtener `shape`, columnas y tipos de datos. Esta metadata permite saber exactamente que version analitica del dataset acompana a los resultados de modelado.

In [ ]:
df_metadata = pd.read_csv(DATA_FILE, nrows=5)
full_shape = pd.read_csv(DATA_FILE).shape

print(f"Dataset procesado: {DATA_FILE.resolve()}")
print(f"Shape: {full_shape}")
display(df_metadata.dtypes.rename("dtype").to_frame())

## Construccion de documentos

Los documentos se construyen a partir de los JSON ya generados en `outputs/`: `model_metrics.json`, `model_comparison.json` y `cross_validation_results.json`. Esta etapa solo persiste resultados existentes; no reentrena modelos.

In [ ]:
payloads = build_firestore_payloads()

print("Documentos datasets:")
display(pd.DataFrame.from_dict(payloads["datasets"], orient="index"))

print("Documentos model_results:")
display(pd.DataFrame.from_dict(payloads["model_results"], orient="index"))

print("Documentos cross_validation:")
display(pd.DataFrame.from_dict(payloads["cross_validation"], orient="index"))

print("Documentos model_config:")
display(pd.DataFrame.from_dict(payloads["model_config"], orient="index"))

print("Eventos de logs:")
display(pd.DataFrame.from_dict(payloads["logs"], orient="index"))


## Carga a Firestore

La carga real requiere credenciales locales. No deben subirse al repo. Configure un archivo `.env` ignorado por Git con `FIREBASE_CREDENTIALS_PATH` o use `GOOGLE_APPLICATION_CREDENTIALS` como variable de entorno.

Por seguridad, `RUN_UPLOAD` queda en `False`. Para ejecutar la escritura desde notebook, cambiarlo a `True` en un entorno local con credenciales configuradas.

In [ ]:
RUN_UPLOAD = False

logger = get_pipeline_logger(LOG_FILE)
logger.info("Inicio de carga a Firebase/Firestore desde notebook")
load_env_file(PROJECT_ROOT / ".env")

if RUN_UPLOAD:
    upload_payloads(payloads, logger)
    logger.info("Carga a Firebase/Firestore desde notebook finalizada correctamente")
else:
    logger.info("Notebook ejecutado en modo revision; no se escribio en Firestore")
    print("Modo revision: no se escribio en Firestore.")

## Ejecucion recomendada por script

Para produccion local se recomienda ejecutar el script versionado:

```bash
python scripts/upload_results_firebase.py --dry-run
python scripts/upload_results_firebase.py
```

El primer comando valida archivos y payloads sin escribir. El segundo crea o actualiza los documentos en Firestore.